# 第 4 章：把注意力装成完整 GPT

上一章的多头注意力只是一个零件。本章把它与归一化、前馈网络和残差连接组成
`TransformerBlock`，再堆叠多个 block，最后由 `out_head` 输出词表预测值。


In [ ]:
from pathlib import Path
import sys

# notebook 在项目根目录或章节目录中打开时，都能找到我们的 Python 文件。
ROOT = Path.cwd()
if not (ROOT / "chapter02").exists():
    ROOT = ROOT.parent
if not (ROOT / "chapter02").exists():
    raise FileNotFoundError("请从本项目的根目录或章节目录启动 Jupyter")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("项目目录:", ROOT)


## 1. 归一化、非线性和前馈网络

LayerNorm 沿一个 token 的向量列做标准化。GELU 让网络能学习非线性变化。
FFN 的形状是 `D → 4D → D`，它对每个 token 使用同一套参数，不在这一步交换 token 信息。


In [ ]:
import torch
from chapter04.gpt import (
    LayerNorm, GELU, FeedForward, TransformerBlock,
    GPTModel, TINY_CONFIG, GPT_CONFIG_124M, generate_text_simple
)

torch.manual_seed(123)
cfg = TINY_CONFIG.copy()
toy = torch.randn(2, 5, cfg["emb_dim"])
print("LayerNorm:", LayerNorm(cfg["emb_dim"])(toy).shape)
print("GELU:", GELU()(toy).shape)
print("FFN:", FeedForward(cfg)(toy).shape)


## 2. 为什么写 `x + F(x)`？

可以把 `F(x)` 想成这一层提出的修改。加法让原信息直接到下一层，
而在反向传播时也给梯度一条直接路径。下面用最简单的 `F(x)=2x` 看数值：
`y=x+2x=3x`，所以梯度 `dy/dx=3`。在真实 GPT 中，`F` 是注意力或 FFN。


In [ ]:
small_x = torch.tensor([1.0, 2.0], requires_grad=True)
small_y = small_x + 2 * small_x
small_y.sum().backward()
print("y:", small_y.detach().tolist())
print("x 的梯度:", small_x.grad.tolist())
block = TransformerBlock(cfg)
print("一个 block 的输入与输出:", toy.shape, block(toy).shape)


## 3. 堆叠 block，再做词表预测

token id 的输入形状是 `[B,T]`。两个 embedding 相加后变为 `[B,T,D]`；
每个 block 仍输出 `[B,T,D]`；`out_head` 最后给每个位置 `V` 个 logits。
`logits` 是未归一化的预测值，第 5 章用交叉熵训练。
这个模型只有**因果 self-attention**，因此属于 decoder-only GPT。


In [ ]:
model = GPTModel(cfg)
input_ids = torch.randint(0, cfg["vocab_size"], (2, 5))
logits = model(input_ids)
print("输入:", input_ids.shape, "输出 [B,T,V]:", logits.shape)
print("小模型参数数:", sum(p.numel() for p in model.parameters()))
print("原书 GPT-2 small 配置的 block 数:", GPT_CONFIG_124M["n_layers"])


## 4. 一次只生成一个 token

生成时取最后一个位置的 logits，选一个 token id，接在输入后再运行。
所有 block 逐层加工表示；最终预测由 `out_head` 完成。
下面模型还没训练，所以输出的 id 没有语言意义。


In [ ]:
model.eval()  # 关闭训练用 dropout
prompt_ids = input_ids[:1, :3]
generated_ids = generate_text_simple(
    model, prompt_ids, max_new_tokens=3, context_size=cfg["context_length"]
)
print("输入 id:", prompt_ids.tolist())
print("生成后 id:", generated_ids.tolist())


**位置方案提醒**：本书与 GPT-2 用 `nn.Embedding` 学每个绝对位置的向量。
原始 Transformer 论文用固定的正弦/余弦公式。二者都提供位置信息，
但前者的表项需要训练；后者的数值由公式直接算出。
